In [ ]:
# Bound the kernel to available RAM. Evaluation loads the shared test slice once,
# then loads/predicts/releases only the eight models for the active horizon.
import os
import sys

sys.path.insert(0, os.path.abspath("../../../Generic-Parallel-Compute-Helper/"))
from memory_compute import *

install_memory_guard()


In [ ]:
import gc
import os
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import pyarrow.parquet as pq
from IPython.display import display
from sklearn.metrics import (
    average_precision_score, brier_score_loss, mean_absolute_error,
    mean_squared_error, r2_score, roc_auc_score,
)
from tqdm.auto import tqdm

from post_training_tail import apply_spike_router

sys.path.append(os.path.abspath("../../"))
from EPF import variables

sys.path.append(os.path.abspath("../2_Features_build"))
import target_features


# Test-set benchmark: central and tail-aware models vs external baselines

For every 30-minute lead from +0.5 to +48 hours this notebook evaluates:

1. **Central stacker** — the MAE-focused point forecast from notebook 3.
2. **Tail-aware forecast** — the calibrated spike alert routed toward a spike
   magnitude expert, with a separately exported spike probability and flag.
3. **Naive persistence** — the latest completed 30-minute mean spot price,
   repeated across the forecast horizon.
4. **AEMO predispatch** — the latest genuinely available predispatch RRP curve.

Targets are sliding 30-minute means generated every five minutes. AEMO
predispatch periods are clock-aligned half-hours, so for horizons h1..h77 the
benchmark is a duration-weighted mixture of the two AEMO periods overlapping
the target window. Horizon h78 is available only at half-hour-aligned origins;
h79..h96 are reported as unavailable because the stored AEMO curve ends at h78.
Missing AEMO forecasts remain missing—there is no unlimited forward filling in
the benchmark.


In [ ]:
SELECTED_FEATURES_DIR = variables.CWD / "4_Features_select" / "Selected_features"
RESULTS_DIR = variables.CWD / "5_Model" / "Data" / "5_model_results"
BENCHMARK_METRICS_PATH = RESULTS_DIR / "forecast_benchmark_metrics.csv"
BENCHMARK_SUMMARY_PATH = RESULTS_DIR / "forecast_benchmark_by_horizon.csv"
AEMO_PREDISPATCH_PATH = (
    variables.CWD / "1_Dataset" / "Processed_data"
    / "6_1_predispatch_price.parquet"
)

FEATURE_OBJECTIVES = ["normal", "arcsinh", "spikes", "dips"]
EXPECTED_HORIZONS = list(range(1, variables.HORIZON_COUNT + 1))
AEMO_MAX_HORIZON = 78

COMPONENT_SPECS = [
    ("base_l1", "normal", "full_range_regressor_clipped_MAE_loss", "regression"),
    ("base_l2", "arcsinh", "full_range_regressor_unclipped_RMSE_loss", "regression"),
    ("spike_probability", "spikes", "positive_spike_classifier_binary_loss", "probability"),
    ("spike_mae", "spikes", "positive_spike_regressor_unclipped_mae_loss", "regression"),
    ("spike_q90", "spikes", "positive_spike_regressor_unclipped_quantile_loss", "regression"),
    ("dip_probability", "dips", "negative_spike_classifer_unclipped_binary_loss", "probability"),
    ("dip_mae", "dips", "negative_spike_regressor_unclipped_mae_loss", "regression"),
    ("dip_q10", "dips", "negative_spike_regressor_unclipped_quantile_loss", "regression"),
]

META_FEATURE_NAMES = [
    "base_l1", "base_l2", "spike_mae", "spike_q90", "dip_mae", "dip_q10",
    "spike_probability", "dip_probability", "spike_logit", "dip_logit",
    "base_l2_minus_l1", "spike_mae_positive_gap", "spike_q90_positive_gap",
    "dip_mae_negative_gap", "dip_q10_negative_gap",
    "weighted_spike_mae_gap", "weighted_spike_q90_gap",
    "weighted_dip_mae_gap", "weighted_dip_q10_gap",
]

THREADS_PER_MODEL = max(1, (psutil.cpu_count(logical=False) or 2) - 2)


In [ ]:
def parquet_data_columns(path):
    schema = pq.ParquetFile(path).schema_arrow
    metadata = schema.pandas_metadata or {}
    index_columns = {
        column for column in metadata.get("index_columns", [])
        if isinstance(column, str)
    }
    return [column for column in schema.names if column not in index_columns]


def read_parquet_date_slice(path, columns, start, end):
    """Read selected columns for `(start, end]` in bounded batches."""
    parquet = pq.ParquetFile(path, pre_buffer=False, memory_map=False)
    metadata = parquet.schema_arrow.pandas_metadata or {}
    index_columns = [
        column for column in metadata.get("index_columns", [])
        if isinstance(column, str)
    ]
    if len(index_columns) != 1:
        raise ValueError(f"Expected one stored index column in {path}; found {index_columns}")
    index_column = index_columns[0]
    available = set(parquet_data_columns(path))
    missing = sorted(set(columns).difference(available))
    if missing:
        raise KeyError(f"{path} is missing requested columns; examples={missing[:5]}")

    parts = []
    for record_batch in parquet.iter_batches(
        batch_size=10_000,
        columns=list(columns) + [index_column],
        use_threads=False,
    ):
        batch = record_batch.to_pandas()
        dates = batch.index if batch.index.name == index_column else batch[index_column]
        mask = (dates > start) & (dates <= end)
        if mask.any():
            batch = batch.loc[mask]
            if index_column in batch.columns:
                batch = batch.set_index(index_column)
            parts.append(batch.astype(np.float32))

    if not parts:
        return pd.DataFrame(columns=columns, index=pd.DatetimeIndex([], name=index_column))
    result = pd.concat(parts).sort_index()
    if result.index.has_duplicates:
        raise ValueError(f"Duplicate timestamps found in {path}")
    return result


def file_signature(paths):
    signature = []
    for path in sorted(map(Path, paths), key=lambda item: item.name):
        if not path.exists():
            raise FileNotFoundError(path)
        stat = path.stat()
        signature.append((path.name, int(stat.st_size), int(stat.st_mtime_ns)))
    return signature


def trained_model_path(horizon, model_name):
    return Path(variables.TRAINED_MODELS_PATH) / f"h{horizon:02d}_{model_name}.joblib"


artifact = joblib.load(variables.FINAL_PARAMS_PATH)
if artifact.get("schema_version") != 3:
    raise ValueError("FINAL_PARAMS_PATH is not a version-3 spike-router artifact; run notebook 3")
if artifact.get("approach") != "purged_residual_stacking_with_calibrated_spike_router":
    raise ValueError(f"Unexpected combiner approach: {artifact.get('approach')}")
if artifact.get("target_region") != variables.TARGET_REGION:
    raise ValueError("Combiner target region does not match variables.TARGET_REGION")
if artifact.get("horizon_list") != EXPECTED_HORIZONS:
    raise ValueError("Combiner does not contain exactly the expected 96 horizons")
if artifact.get("component_specs") != COMPONENT_SPECS:
    raise ValueError("Component model layout differs from the fitted combiner")
if artifact.get("meta_feature_names") != META_FEATURE_NAMES:
    raise ValueError("Meta-feature layout differs from the fitted combiner")
if len(artifact.get("stackers", [])) != len(EXPECTED_HORIZONS):
    raise ValueError("Combiner artifact has an incomplete stacker list")
if len(artifact.get("spike_routers", [])) != len(EXPECTED_HORIZONS):
    raise ValueError("Combiner artifact has an incomplete spike-router list")

all_model_paths = [
    trained_model_path(horizon, model_name)
    for horizon in EXPECTED_HORIZONS
    for _, _, model_name, _ in COMPONENT_SPECS
]
selection_paths = [
    SELECTED_FEATURES_DIR / f"FEATURES_OPTIMAL_AMOUNT_{objective}.parquet"
    for objective in FEATURE_OBJECTIVES
]
if file_signature(all_model_paths) != artifact["trained_model_signature"]:
    raise RuntimeError("Trained models changed after notebook 3 built the stackers")
if file_signature(selection_paths) != artifact["selected_feature_signature"]:
    raise RuntimeError("Selected-feature metadata changed after notebook 3 built the stackers")

print(
    f"Loaded {len(artifact['stackers'])} {artifact['approach']} models "
    f"created {artifact['created_utc']}"
)


In [ ]:
features_optimal_amount_by_objective = {
    objective: pd.read_parquet(
        SELECTED_FEATURES_DIR / f"FEATURES_OPTIMAL_AMOUNT_{objective}.parquet"
    )
    for objective in FEATURE_OBJECTIVES
}


def selected_features(objective, horizon):
    table = features_optimal_amount_by_objective[objective]
    result = table.loc[table[f"h{horizon}"].astype(bool), "feature"].tolist()
    if not result:
        raise ValueError(f"No selected features for {objective} h{horizon}")
    return result


needed_features = sorted({
    feature
    for objective in FEATURE_OBJECTIVES
    for horizon in EXPECTED_HORIZONS
    for feature in selected_features(objective, horizon)
})
target_columns = [f"target_h{h}" for h in EXPECTED_HORIZONS]

targets_test = read_parquet_date_slice(
    variables.AGG_TARGET_DATASET_PATH,
    target_columns,
    variables.TEST_START,
    variables.PIPELINE_END_DATE,
).dropna(subset=target_columns)
test_index = targets_test.index

features_test = read_parquet_date_slice(
    variables.FEATURES_DATASET_PATH,
    needed_features,
    variables.TEST_START,
    variables.PIPELINE_END_DATE,
).reindex(test_index)
if features_test.isna().any().any():
    bad = features_test.columns[features_test.isna().any()].tolist()
    raise ValueError(f"Test feature slice contains NaNs; examples={bad[:5]}")

# Naive persistence: latest completed 30-minute mean of observed 5-minute RRP.
spot_price = pd.read_parquet(
    variables.TARGET_DATASET_PATH,
    columns=[variables.SELECTED_TARGET_COLUMN_NAME],
)[variables.SELECTED_TARGET_COLUMN_NAME].astype(np.float32)
periods_per_target = (
    variables.HORIZON_GRANULARITY_IN_MINUTES
    // variables.FEATURE_GRANULARITY_IN_MINUTES
)
naive_prediction = (
    spot_price.rolling(periods_per_target, min_periods=periods_per_target)
    .mean()
    .reindex(test_index)
    .to_numpy(dtype=np.float32)
)
if not np.isfinite(naive_prediction).all():
    raise ValueError("Naive persistence forecast contains missing values")

# Use the unfilled processed source, not 0_all_features, so genuine AEMO gaps
# remain visible in availability counts and cannot become multi-day stale values.
aemo_requested_columns = [
    f"predispatch_rrp_{variables.TARGET_REGION}_h{h}"
    for h in range(1, AEMO_MAX_HORIZON + 1)
]
aemo_available_columns = set(parquet_data_columns(AEMO_PREDISPATCH_PATH))
aemo_columns = [c for c in aemo_requested_columns if c in aemo_available_columns]
aemo_test = read_parquet_date_slice(
    AEMO_PREDISPATCH_PATH,
    aemo_columns,
    variables.TEST_START,
    variables.PIPELINE_END_DATE,
).reindex(test_index)

print(
    f"Test data: {len(test_index):,} origins, {len(EXPECTED_HORIZONS)} horizons, "
    f"{len(needed_features):,} selected features, {len(aemo_columns)} AEMO curve columns"
)


In [ ]:
def convert_from_asinh(values):
    return (np.sinh(values) * variables.PRICE_TRANSFORM_SCALE).astype(np.float32)


def predict_components(horizon, feature_frame):
    predictions = {}
    for objective in FEATURE_OBJECTIVES:
        columns = selected_features(objective, horizon)
        X_horizon = target_features.append_target_time_feats(
            feature_frame[columns].astype(np.float32), horizon
        )
        for key, spec_objective, model_name, output_kind in COMPONENT_SPECS:
            if spec_objective != objective:
                continue
            model = joblib.load(trained_model_path(horizon, model_name))
            if hasattr(model, "set_params"):
                model.set_params(
                    n_jobs=THREADS_PER_MODEL,
                    num_threads=THREADS_PER_MODEL,
                )
            if int(model.n_features_in_) != X_horizon.shape[1]:
                raise ValueError(
                    f"{model_name} h{horizon} expects {model.n_features_in_} features; "
                    f"received {X_horizon.shape[1]}"
                )
            if output_kind == "probability":
                values = model.predict_proba(X_horizon)[:, 1].astype(np.float32)
            else:
                values = convert_from_asinh(model.predict(X_horizon))
            if not np.isfinite(values).all():
                raise ValueError(f"Non-finite predictions from {model_name} h{horizon}")
            predictions[key] = values
            del model
        del X_horizon
        gc.collect()
    return predictions


def build_meta_features(component_predictions):
    base_l1 = component_predictions["base_l1"]
    base_l2 = component_predictions["base_l2"]
    spike_mae = component_predictions["spike_mae"]
    spike_q90 = component_predictions["spike_q90"]
    dip_mae = component_predictions["dip_mae"]
    dip_q10 = component_predictions["dip_q10"]
    epsilon = np.float32(1e-6)
    spike_probability = np.clip(
        component_predictions["spike_probability"], epsilon, 1.0 - epsilon
    )
    dip_probability = np.clip(
        component_predictions["dip_probability"], epsilon, 1.0 - epsilon
    )
    spike_logit = np.log(spike_probability / (1.0 - spike_probability))
    dip_logit = np.log(dip_probability / (1.0 - dip_probability))
    spike_mae_gap = np.maximum(spike_mae - base_l1, 0.0)
    spike_q90_gap = np.maximum(spike_q90 - base_l1, 0.0)
    dip_mae_gap = np.minimum(dip_mae - base_l1, 0.0)
    dip_q10_gap = np.minimum(dip_q10 - base_l1, 0.0)

    meta = np.column_stack([
        base_l1, base_l2, spike_mae, spike_q90, dip_mae, dip_q10,
        spike_probability, dip_probability, spike_logit, dip_logit,
        base_l2 - base_l1, spike_mae_gap, spike_q90_gap,
        dip_mae_gap, dip_q10_gap,
        spike_probability * spike_mae_gap,
        spike_probability * spike_q90_gap,
        dip_probability * dip_mae_gap,
        dip_probability * dip_q10_gap,
    ]).astype(np.float32)
    if meta.shape[1] != len(META_FEATURE_NAMES) or not np.isfinite(meta).all():
        raise ValueError("Invalid meta-feature matrix")
    return meta, base_l1


def predict_postprocessed_horizon(horizon):
    components = predict_components(horizon, features_test)
    meta, anchor = build_meta_features(components)
    stacker = artifact["stackers"][horizon - 1]
    if int(stacker.n_features_in_) != meta.shape[1]:
        raise ValueError(f"Stacker h{horizon} feature-count mismatch")
    central_prediction = (anchor + stacker.predict(meta)).astype(np.float32)
    if not np.isfinite(central_prediction).all():
        raise ValueError(f"Stacker h{horizon} produced non-finite predictions")
    tail_prediction, spike_probability, spike_alert = apply_spike_router(
        central_prediction,
        components,
        artifact["spike_routers"][horizon - 1],
        variables.SPIKE_THRESHOLD,
    )
    if not np.isfinite(tail_prediction).all() or not np.isfinite(spike_probability).all():
        raise ValueError(f"Spike router h{horizon} produced non-finite predictions")
    del components, meta, anchor, stacker
    gc.collect()
    return central_prediction, tail_prediction, spike_probability, spike_alert


In [ ]:
def aligned_aemo_prediction(horizon):
    """Match fixed AEMO half-hours to this pipeline's sliding 30-minute target."""
    result = np.full(len(test_index), np.nan, dtype=np.float32)
    current_column = f"predispatch_rrp_{variables.TARGET_REGION}_h{horizon}"
    if horizon > AEMO_MAX_HORIZON or current_column not in aemo_test:
        return result

    current = aemo_test[current_column].to_numpy(dtype=np.float32)
    offset_steps = ((test_index.minute % 30) // variables.FEATURE_GRANULARITY_IN_MINUTES).astype(int)
    next_weight = offset_steps.astype(np.float32) / periods_per_target
    aligned = offset_steps == 0
    result[aligned] = current[aligned]

    next_column = f"predispatch_rrp_{variables.TARGET_REGION}_h{horizon + 1}"
    crossing = ~aligned
    if crossing.any() and next_column in aemo_test:
        following = aemo_test[next_column].to_numpy(dtype=np.float32)
        result[crossing] = (
            (1.0 - next_weight[crossing]) * current[crossing]
            + next_weight[crossing] * following[crossing]
        )
    return result


def error_metrics(actual, predicted, predicted_spike_override=None):
    valid = np.isfinite(actual) & np.isfinite(predicted)
    n = int(valid.sum())
    if n == 0:
        return {
            "n": 0, "availability_pct": 0.0,
            "mae": np.nan, "rmse": np.nan, "r2": np.nan,
            "mbe": np.nan, "wmape": np.nan,
            "spike_mae": np.nan, "dip_mae": np.nan, "normal_mae": np.nan,
            "spike_precision": np.nan, "spike_recall": np.nan, "spike_f2": np.nan,
            "dip_precision": np.nan, "dip_recall": np.nan,
        }

    y = actual[valid].astype(np.float64)
    p = predicted[valid].astype(np.float64)
    error = p - y
    spike = y > variables.SPIKE_THRESHOLD
    dip = y < variables.DIP_THRESHOLD
    normal = ~(spike | dip)
    if predicted_spike_override is None:
        predicted_spike = p > variables.SPIKE_THRESHOLD
    else:
        predicted_spike = np.asarray(predicted_spike_override, dtype=bool)[valid]
    predicted_dip = p < variables.DIP_THRESHOLD

    def subset_mae(mask):
        return float(np.mean(np.abs(error[mask]))) if mask.any() else np.nan

    def precision_recall(actual_mask, predicted_mask):
        true_positive = int(np.count_nonzero(actual_mask & predicted_mask))
        predicted_positive = int(np.count_nonzero(predicted_mask))
        actual_positive = int(np.count_nonzero(actual_mask))
        precision = true_positive / predicted_positive if predicted_positive else np.nan
        recall = true_positive / actual_positive if actual_positive else np.nan
        return precision, recall

    spike_precision, spike_recall = precision_recall(spike, predicted_spike)
    dip_precision, dip_recall = precision_recall(dip, predicted_dip)
    spike_f2 = (
        5.0 * spike_precision * spike_recall / (4.0 * spike_precision + spike_recall)
        if np.isfinite(spike_precision) and np.isfinite(spike_recall)
        and spike_precision + spike_recall else np.nan
    )
    return {
        "n": n,
        "availability_pct": 100.0 * n / len(actual),
        "mae": float(mean_absolute_error(y, p)),
        "rmse": float(np.sqrt(mean_squared_error(y, p))),
        "r2": float(r2_score(y, p)) if n >= 2 else np.nan,
        "mbe": float(np.mean(error)),
        "wmape": float(100.0 * np.sum(np.abs(error)) / (np.sum(np.abs(y)) + 1e-8)),
        "spike_mae": subset_mae(spike),
        "dip_mae": subset_mae(dip),
        "normal_mae": subset_mae(normal),
        "spike_precision": spike_precision,
        "spike_recall": spike_recall,
        "spike_f2": spike_f2,
        "dip_precision": dip_precision,
        "dip_recall": dip_recall,
    }


def skill_pct(reference_error, model_error):
    if not np.isfinite(reference_error) or reference_error == 0:
        return np.nan
    return 100.0 * (reference_error - model_error) / reference_error


## Generate predictions and calculate every horizon benchmark

The loop is intentionally sequential. It retains compact result matrices,
but releases component models and horizon-specific feature frames immediately.


In [ ]:
n_rows = len(test_index)
n_horizons = len(EXPECTED_HORIZONS)
actual_matrix = targets_test[target_columns].to_numpy(dtype=np.float32)
model_matrix = np.full((n_rows, n_horizons), np.nan, dtype=np.float32)
tail_matrix = np.full((n_rows, n_horizons), np.nan, dtype=np.float32)
spike_probability_matrix = np.full((n_rows, n_horizons), np.nan, dtype=np.float32)
spike_alert_matrix = np.zeros((n_rows, n_horizons), dtype=bool)
naive_matrix = np.repeat(naive_prediction[:, None], n_horizons, axis=1)
aemo_matrix = np.full((n_rows, n_horizons), np.nan, dtype=np.float32)

metric_records = []
horizon_records = []

for column_index, horizon in enumerate(
    tqdm(EXPECTED_HORIZONS, desc="Evaluating horizons", unit="horizon")
):
    actual = actual_matrix[:, column_index]
    (
        model_prediction,
        tail_prediction,
        spike_probability,
        spike_alert,
    ) = predict_postprocessed_horizon(horizon)
    aemo_prediction = aligned_aemo_prediction(horizon)
    model_matrix[:, column_index] = model_prediction
    tail_matrix[:, column_index] = tail_prediction
    spike_probability_matrix[:, column_index] = spike_probability
    spike_alert_matrix[:, column_index] = spike_alert
    aemo_matrix[:, column_index] = aemo_prediction

    method_predictions = {
        "residual_stacker": (model_prediction, None),
        "tail_aware_spike_router": (tail_prediction, spike_alert),
        "naive_persistence": (naive_prediction, None),
        "aemo_predispatch": (aemo_prediction, None),
    }
    per_method = {}
    for method, (prediction, spike_override) in method_predictions.items():
        values = error_metrics(actual, prediction, spike_override)
        per_method[method] = values
        metric_records.append({
            "horizon": horizon,
            "lead_hours": horizon * variables.HORIZON_GRANULARITY_IN_MINUTES / 60.0,
            "method": method,
            **values,
        })

    # Model-vs-AEMO skill is calculated on AEMO's exact available sample.
    aemo_valid = np.isfinite(actual) & np.isfinite(aemo_prediction)
    if aemo_valid.any():
        aemo_common = error_metrics(actual[aemo_valid], aemo_prediction[aemo_valid])
        model_common = error_metrics(actual[aemo_valid], model_prediction[aemo_valid])
        tail_common = error_metrics(
            actual[aemo_valid], tail_prediction[aemo_valid], spike_alert[aemo_valid]
        )
        naive_common = error_metrics(actual[aemo_valid], naive_prediction[aemo_valid])
    else:
        aemo_common = {"mae": np.nan, "rmse": np.nan}
        model_common = {"mae": np.nan, "rmse": np.nan}
        tail_common = {"mae": np.nan, "rmse": np.nan}
        naive_common = {"mae": np.nan, "rmse": np.nan}

    horizon_records.append({
        "horizon": horizon,
        "lead_hours": horizon * variables.HORIZON_GRANULARITY_IN_MINUTES / 60.0,
        "model_mae": per_method["residual_stacker"]["mae"],
        "tail_mae": per_method["tail_aware_spike_router"]["mae"],
        "naive_mae": per_method["naive_persistence"]["mae"],
        "aemo_mae": per_method["aemo_predispatch"]["mae"],
        "model_rmse": per_method["residual_stacker"]["rmse"],
        "tail_rmse": per_method["tail_aware_spike_router"]["rmse"],
        "naive_rmse": per_method["naive_persistence"]["rmse"],
        "aemo_rmse": per_method["aemo_predispatch"]["rmse"],
        "aemo_availability_pct": per_method["aemo_predispatch"]["availability_pct"],
        "model_mae_skill_vs_naive_pct": skill_pct(
            per_method["naive_persistence"]["mae"],
            per_method["residual_stacker"]["mae"],
        ),
        "model_rmse_skill_vs_naive_pct": skill_pct(
            per_method["naive_persistence"]["rmse"],
            per_method["residual_stacker"]["rmse"],
        ),
        "tail_mae_change_vs_central_pct": skill_pct(
            per_method["residual_stacker"]["mae"],
            per_method["tail_aware_spike_router"]["mae"],
        ),
        "central_spike_mae": per_method["residual_stacker"]["spike_mae"],
        "tail_spike_mae": per_method["tail_aware_spike_router"]["spike_mae"],
        "tail_spike_mae_skill_vs_central_pct": skill_pct(
            per_method["residual_stacker"]["spike_mae"],
            per_method["tail_aware_spike_router"]["spike_mae"],
        ),
        "central_spike_precision": per_method["residual_stacker"]["spike_precision"],
        "central_spike_recall": per_method["residual_stacker"]["spike_recall"],
        "tail_spike_precision": per_method["tail_aware_spike_router"]["spike_precision"],
        "tail_spike_recall": per_method["tail_aware_spike_router"]["spike_recall"],
        "central_spike_f2": per_method["residual_stacker"]["spike_f2"],
        "tail_spike_f2": per_method["tail_aware_spike_router"]["spike_f2"],
        "tail_spike_recall_gain_pp": 100.0 * (
            per_method["tail_aware_spike_router"]["spike_recall"]
            - per_method["residual_stacker"]["spike_recall"]
        ),
        "spike_probability_average_precision": float(
            average_precision_score(actual > variables.SPIKE_THRESHOLD, spike_probability)
        ),
        "spike_probability_roc_auc": float(
            roc_auc_score(actual > variables.SPIKE_THRESHOLD, spike_probability)
        ),
        "spike_probability_brier": float(
            brier_score_loss(actual > variables.SPIKE_THRESHOLD, spike_probability)
        ),
        "model_mae_on_aemo_sample": model_common["mae"],
        "tail_mae_on_aemo_sample": tail_common["mae"],
        "naive_mae_on_aemo_sample": naive_common["mae"],
        "model_mae_skill_vs_aemo_pct": skill_pct(
            aemo_common["mae"], model_common["mae"]
        ),
        "tail_mae_skill_vs_aemo_pct": skill_pct(
            aemo_common["mae"], tail_common["mae"]
        ),
    })

benchmark_metrics = pd.DataFrame(metric_records)
benchmark_by_horizon = pd.DataFrame(horizon_records)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
benchmark_metrics.to_csv(BENCHMARK_METRICS_PATH, index=False)
benchmark_by_horizon.to_csv(BENCHMARK_SUMMARY_PATH, index=False)

actuals_df = pd.DataFrame(
    actual_matrix,
    index=test_index,
    columns=[f"actual_h{h}" for h in EXPECTED_HORIZONS],
)
model_df = pd.DataFrame(
    model_matrix,
    index=test_index,
    columns=[f"predicted_h{h}" for h in EXPECTED_HORIZONS],
)
tail_df = pd.DataFrame(
    tail_matrix,
    index=test_index,
    columns=[f"tail_aware_h{h}" for h in EXPECTED_HORIZONS],
)
spike_probability_df = pd.DataFrame(
    spike_probability_matrix,
    index=test_index,
    columns=[f"spike_probability_h{h}" for h in EXPECTED_HORIZONS],
)
spike_alert_df = pd.DataFrame(
    spike_alert_matrix,
    index=test_index,
    columns=[f"spike_alert_h{h}" for h in EXPECTED_HORIZONS],
)
naive_df = pd.DataFrame(
    naive_matrix,
    index=test_index,
    columns=[f"naive_h{h}" for h in EXPECTED_HORIZONS],
)
aemo_df = pd.DataFrame(
    aemo_matrix,
    index=test_index,
    columns=[f"aemo_predispatch_h{h}" for h in EXPECTED_HORIZONS],
)
combined_df = pd.concat(
    [
        actuals_df, model_df, tail_df, spike_probability_df, spike_alert_df,
        naive_df, aemo_df,
    ],
    axis=1,
)
combined_df.index.name = "date"
combined_df.to_parquet(variables.ACTUAL_VS_PREDICTED_TEST_SET)

print(f"Saved predictions to {variables.ACTUAL_VS_PREDICTED_TEST_SET}")
print(f"Saved long-form metrics to {BENCHMARK_METRICS_PATH}")
print(f"Saved horizon comparison to {BENCHMARK_SUMMARY_PATH}")


In [ ]:
print("Pooled test-set metrics")
pooled_records = []
for method, matrix in {
    "residual_stacker": model_matrix,
    "tail_aware_spike_router": tail_matrix,
    "naive_persistence": naive_matrix,
    "aemo_predispatch": aemo_matrix,
}.items():
    values = error_metrics(actual_matrix.ravel(), matrix.ravel())
    pooled_records.append({"method": method, **values})
pooled_metrics = pd.DataFrame(pooled_records)
display(pooled_metrics)

print("Per-horizon comparison")
display(benchmark_by_horizon)

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
for method, label in [
    ("residual_stacker", "Residual stacker"),
    ("tail_aware_spike_router", "Tail-aware forecast"),
    ("naive_persistence", "Naive persistence"),
    ("aemo_predispatch", "AEMO predispatch"),
]:
    subset = benchmark_metrics[benchmark_metrics["method"] == method]
    axes[0].plot(subset["lead_hours"], subset["mae"], label=label)
    axes[1].plot(subset["lead_hours"], subset["rmse"], label=label)

axes[0].set_ylabel("MAE ($/MWh)")
axes[1].set_ylabel("RMSE ($/MWh)")
axes[1].set_xlabel("Forecast lead (hours)")
axes[0].set_title("Forecast accuracy by horizon")
for axis in axes:
    axis.grid(alpha=0.25)
    axis.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Free this kernel's memory after all outputs have been written.
release_memory()
